# Kiểm thử Lớp C4 — sinh ảnh AI + đánh giá tổng thể

**Trước khi chạy notebook này:**
1. Thêm `CLOUDFLARE_ACCOUNT_ID=...` và `CLOUDFLARE_API_TOKEN=...` vào file `.env` ở gốc project (token cần quyền "Workers AI - Edit", tạo tại dash.cloudflare.com -> My Profile -> API Tokens).
2. Chạy `npm run dev` (server mặc định ở `http://localhost:3000`).
3. Đảm bảo đã chạy `npx prisma db seed` (bảng `TrangPhuc`, `MauSac`, `PhuKien`, `SuKien` có dữ liệu).

**Mục tiêu:** gọi API `POST /api/to-hop-duoc-duyet` (Lớp C4) với **5 tổ hợp** trước,
để phát hiện lỗi prompt / lỗi cấu hình Cloudflare sớm — mỗi lỗi phát hiện ở đây
không tốn thêm lượt gọi ảnh, vì route đã idempotent theo `comboKey`
(gọi lại 1 tổ hợp đã tồn tại sẽ KHÔNG gọi lại AI).

Chỉ khi 5 tổ hợp thử nghiệm chạy ổn (không lỗi cấu hình/lỗi prompt), và đã duyệt
thử qua `PATCH /api/to-hop-duoc-duyet/{id}`, mới chạy **hàng loạt** ở cell cuối.


In [1]:
import requests
import itertools
import time
import json

BASE_URL = "http://localhost:3000"

def lay_danh_sach(duong_dan: str):
    r = requests.get(f"{BASE_URL}{duong_dan}", timeout=10)
    r.raise_for_status()
    return r.json()["data"]


In [2]:
# Tải dữ liệu tham chiếu từ chính API đã có (Lớp C1-C3), không hardcode id.
trang_phuc = lay_danh_sach("/api/trang-phuc")
mau_sac = lay_danh_sach("/api/mau-sac")
phu_kien = lay_danh_sach("/api/phu-kien")
su_kien = lay_danh_sach("/api/su-kien")

print(f"Trang phục: {len(trang_phuc)} | Màu sắc: {len(mau_sac)} | "
      f"Phụ kiện: {len(phu_kien)} | Sự kiện: {len(su_kien)}")

assert trang_phuc and mau_sac and phu_kien and su_kien, (
    "Danh mục rỗng — kiểm tra lại đã seed DB (`npx prisma db seed`) và server đang chạy chưa."
)


Trang phục: 3 | Màu sắc: 10 | Phụ kiện: 9 | Sự kiện: 5


## 1. Chọn 5 tổ hợp thử nghiệm

Chọn thủ công, cố tình đa dạng (khác trang phục / khác nhóm màu / có và
không có cảnh báo văn hoá) để dễ phát hiện lỗi prompt ở nhiều tình huống
khác nhau, thay vì 5 tổ hợp gần giống nhau.

In [3]:
def theo_ten(danh_sach, ten):
    for item in danh_sach:
        if item["ten"] == ten:
            return item
    raise ValueError(f'Không tìm thấy "{ten}" trong danh sách — kiểm tra lại seed.')

def to_hop(ten_trang_phuc, ten_mau_chinh, ten_mau_phu, ten_phu_kien, ten_su_kien):
    return {
        "trangPhucId": theo_ten(trang_phuc, ten_trang_phuc)["id"],
        "mauChinhId": theo_ten(mau_sac, ten_mau_chinh)["id"],
        "mauPhuId": theo_ten(mau_sac, ten_mau_phu)["id"],
        "phuKienId": theo_ten(phu_kien, ten_phu_kien)["id"],
        "suKienId": theo_ten(su_kien, ten_su_kien)["id"],
        "_nhan": f"{ten_trang_phuc} + {ten_mau_chinh}/{ten_mau_phu} + {ten_phu_kien} ({ten_su_kien})",
    }

# 5 tổ hợp mẫu — sửa lại tên nếu bộ seed của bạn khác.
to_hop_thu_nghiem = [
    to_hop("Áo dài", "Đỏ son", "Vàng nhũ", "Nón lá", "Tết"),
    to_hop("Áo tứ thân", "Lam chàm", "Trắng ngà", "Nón quai thao", "Lễ hội"),
    to_hop("Áo bà ba", "Lục ngọc", "Be pastel", "Khăn rằn", "Đi học, dạo phố"),
    to_hop("Áo dài", "Tím Huế", "Đen huyền", "Quạt giấy", "Chụp ảnh nghệ thuật"),
    to_hop("Áo tứ thân", "Hồng đào", "Nâu non", "Yếm đào", "Cưới hỏi"),
]

for th in to_hop_thu_nghiem:
    print("-", th["_nhan"])


- Áo dài + Đỏ son/Vàng nhũ + Nón lá (Tết)
- Áo tứ thân + Lam chàm/Trắng ngà + Nón quai thao (Lễ hội)
- Áo bà ba + Lục ngọc/Be pastel + Khăn rằn (Đi học, dạo phố)
- Áo dài + Tím Huế/Đen huyền + Quạt giấy (Chụp ảnh nghệ thuật)
- Áo tứ thân + Hồng đào/Nâu non + Yếm đào (Cưới hỏi)


In [4]:
def goi_sinh_to_hop(th: dict) -> dict:
    """Gọi POST /api/to-hop-duoc-duyet cho 1 tổ hợp, trả về kết quả đã tóm tắt."""
    body = {k: v for k, v in th.items() if k != "_nhan"}
    bat_dau = time.time()
    try:
        r = requests.post(f"{BASE_URL}/api/to-hop-duoc-duyet", json=body, timeout=90)
    except requests.exceptions.RequestException as e:
        return {"nhan": th["_nhan"], "loi": f"Không gọi được server: {e}"}

    thoi_gian = round(time.time() - bat_dau, 1)
    try:
        payload = r.json()
    except ValueError:
        payload = {"error": r.text[:300]}

    if r.status_code >= 400:
        return {
            "nhan": th["_nhan"],
            "http_status": r.status_code,
            "loi": payload.get("error"),
            "chi_tiet": payload.get("chiTiet"),
            "thoi_gian_s": thoi_gian,
        }

    du_lieu = payload["data"]
    return {
        "nhan": th["_nhan"],
        "http_status": r.status_code,
        "da_ton_tai": payload.get("daTonTai"),
        "comboKey": du_lieu["comboKey"],
        "status": du_lieu["status"],
        "imageUrl": du_lieu.get("imageUrl"),
        "nhanXetAI": (du_lieu.get("aiAssessment") or {}).get("nhanXetAI"),
        "thoi_gian_s": thoi_gian,
    }


In [5]:
ket_qua_thu_nghiem = []
for th in to_hop_thu_nghiem:
    kq = goi_sinh_to_hop(th)
    ket_qua_thu_nghiem.append(kq)
    trang_thai = "LỖI" if "loi" in kq and kq["loi"] else "OK"
    print(f"[{trang_thai}] {kq['nhan']} ({kq.get('thoi_gian_s', '?')}s)")
    if "loi" in kq and kq["loi"]:
        print("       ->", kq["loi"])
        if kq.get("chi_tiet"):
            print("       chi tiết:", json.dumps(kq["chi_tiet"], ensure_ascii=False)[:400])


[OK] Áo dài + Đỏ son/Vàng nhũ + Nón lá (Tết) (1.7s)
[OK] Áo tứ thân + Lam chàm/Trắng ngà + Nón quai thao (Lễ hội) (1.4s)
[OK] Áo bà ba + Lục ngọc/Be pastel + Khăn rằn (Đi học, dạo phố) (1.4s)
[OK] Áo dài + Tím Huế/Đen huyền + Quạt giấy (Chụp ảnh nghệ thuật) (1.3s)
[OK] Áo tứ thân + Hồng đào/Nâu non + Yếm đào (Cưới hỏi) (1.4s)


**Đọc kết quả ở trên trước khi đi tiếp:**
- Lỗi `Thiếu CLOUDFLARE_ACCOUNT_ID hoặc CLOUDFLARE_API_TOKEN` → thêm vào `.env`, khởi động lại `npm run dev`.
- HTTP 502 kèm `chi_tiet` → thường là token thiếu quyền "Workers AI - Edit", hoặc prompt bị model từ chối — đọc `chi_tiet` để biết chính xác.
- Ảnh ra không đúng như mô tả (sai loại trang phục/phụ kiện) → đây là hạn chế đã biết của flux-1-schnell với mô tả tiếng Việt, thử chỉnh `taoPromptAnh()` trong `lib/aiSinhAnh.ts` cho chi tiết hơn (mô tả dáng, chất liệu bằng tiếng Anh) rồi chạy lại đúng cell trên — không cần chạy lại từ đầu.
- `da_ton_tai: true` → tổ hợp đã có sẵn trong DB, không tốn lượt gọi ảnh (đúng như thiết kế).


In [6]:
from IPython.display import Image, display

for kq in ket_qua_thu_nghiem:
    if kq.get("imageUrl"):
        print(kq["nhan"], "-", kq.get("nhanXetAI"))
        display(Image(url=f"{BASE_URL}{kq['imageUrl']}"))


Áo dài + Đỏ son/Vàng nhũ + Nón lá (Tết) - Tổ hợp phối đồ này là một sự kết hợp tuyệt vời giữa phong cách truyền thống Việt Nam và phong cách Gen Z hiện đại, tạo nên một vẻ đẹp độc đáo và thu hút. Áo dài đỏ son kết hợp với nón lá vàng nhũ tạo nên một phong cách thời trang độc đáo và mang đậm tính cách Việt Nam.


Áo tứ thân + Lam chàm/Trắng ngà + Nón quai thao (Lễ hội) - Tổ hợp phối đồ này đã tái tạo tinh thần truyền thống Việt Nam một cách hiện đại và tươi mới, kết hợp màu lam chàm truyền thống với trắng ngà tinh tế tạo nên một phong cách độc đáo. Áo tứ thân và nón quai thao đã mang đến một diện mạo thời trang mới mẻ và năng động cho lễ hội.


Áo bà ba + Lục ngọc/Be pastel + Khăn rằn (Đi học, dạo phố) - Tổ hợp phối đồ này đã mang đến một phong cách hiện đại và sành điệu cho áo bà ba truyền thống, tạo nên một vẻ ngoài độc đáo và thu hút. Màu sắc be pastel đã bổ trợ hiệu quả cho sắc lục ngọc, tạo nên một sự hài hòa tuyệt vời và thu hút ánh nhìn.


Áo dài + Tím Huế/Đen huyền + Quạt giấy (Chụp ảnh nghệ thuật) - Tổ hợp phối đồ này tạo nên một hình ảnh thời trang độc đáo, hài hòa và đầy tinh tế khi kết hợp áo dài truyền thống với màu tím Huế và đen huyền, tạo điểm nhấn cho bức ảnh nghệ thuật. Quạt giấy nhỏ xinh xinh thêm một lớp tinh tế, tôn lên vẻ đẹp thời trang của người mặc.


Áo tứ thân + Hồng đào/Nâu non + Yếm đào (Cưới hỏi) - Tổ hợp phối đồ này mang lại cảm giác tươi mới và hiện đại với sự kết hợp hài hòa giữa áo tứ thân truyền thống và phụ kiện yếm đào, tạo nên một phong cách độc đáo và thời thượng cho dịp cưới hỏi. Màu sắc hồng đào và nâu non tạo nên một sự đối lập thú vị, mang lại cảm giác độc đáo và cuốn hút.


## 2. Duyệt thử qua route duyệt

Trước khi chạy hàng loạt, thử duyệt (approve) 1 tổ hợp ở trên để chắc chắn
route duyệt hoạt động đúng.

In [7]:
# Lấy danh sách đang chờ duyệt
r = requests.get(f"{BASE_URL}/api/to-hop-duoc-duyet", params={"status": "draft"}, timeout=10)
r.raise_for_status()
dang_cho_duyet = r.json()["data"]
print(f"Đang có {len(dang_cho_duyet)} tổ hợp ở trạng thái draft.")

if dang_cho_duyet:
    item = dang_cho_duyet[0]
    r2 = requests.patch(
        f"{BASE_URL}/api/to-hop-duoc-duyet/{item['id']}",
        json={"status": "approved"},
        timeout=10,
    )
    r2.raise_for_status()
    print("Đã duyệt thử:", r2.json()["data"]["comboKey"], "-> status:", r2.json()["data"]["status"])


Đang có 4 tổ hợp ở trạng thái draft.
Đã duyệt thử: a89345d6-1043-4bdf-b49e-16a31dd280bf:9ada5b64-5bb7-4946-999c-197d1d331c19:e660db95-9565-4e58-870f-8f03cbe8abbc:5b1bbc09-0ba1-4cfa-afb0-c1c358c67536:7b03d604-e08b-4ff4-9846-56ca30fed04c -> status: approved


## 3. Chạy hàng loạt (chỉ chạy sau khi bước 1 & 2 đều ổn)

Sinh toàn bộ tổ hợp còn lại: `trangPhuc x mauChinh x mauPhu (khác nhau) x phuKien x suKien`.
Vì route C4 **idempotent theo comboKey**, chạy lại cell này nhiều lần là an toàn —
các tổ hợp đã tồn tại sẽ trả về ngay (`daTonTai: true`), không tốn thêm lượt gọi ảnh.

Có lọc trước qua `POST /api/kiem-tra-phoi-do` để **bỏ qua** tổ hợp bị cảnh báo
không phù hợp văn hoá (`canhBao: true`) — tránh tốn lượt gọi ảnh cho tổ hợp
chắc chắn sẽ bị từ chối khi duyệt.

In [7]:
CHAY_HANG_LOAT = False  # đổi thành True sau khi đã kiểm tra kỹ 5 tổ hợp + duyệt thử ở trên
GIOI_HAN_SO_LUONG = None  # vd: 20 để chạy thử 1 phần trước khi chạy hết
DO_TRE_GIUA_2_LUOT_GIAY = 1.0  # nghỉ giữa các lượt gọi, tránh dồn dập vào Cloudflare Workers AI

def phu_hop_van_hoa(trang_phuc_id, phu_kien_id, su_kien_id) -> bool:
    r = requests.post(
        f"{BASE_URL}/api/kiem-tra-phoi-do",
        json={
            "trangPhucId": trang_phuc_id,
            "mauChinhId": mau_sac[0]["id"],  # màu không ảnh hưởng đến kiểm tra văn hoá
            "mauPhuId": mau_sac[1]["id"],
            "phuKienId": phu_kien_id,
            "suKienId": su_kien_id,
        },
        timeout=10,
    )
    r.raise_for_status()
    return not r.json()["data"]["phuHopVanHoa"]["canhBao"]


def sinh_tat_ca_to_hop():
    for tp in trang_phuc:
        for pk in phu_kien:
            for sk in su_kien:
                if not phu_hop_van_hoa(tp["id"], pk["id"], sk["id"]):
                    continue  # bỏ qua tổ hợp chắc chắn không phù hợp văn hoá
                for mc, mp in itertools.permutations(mau_sac, 2):
                    yield {
                        "trangPhucId": tp["id"],
                        "mauChinhId": mc["id"],
                        "mauPhuId": mp["id"],
                        "phuKienId": pk["id"],
                        "suKienId": sk["id"],
                        "_nhan": f"{tp['ten']} + {mc['ten']}/{mp['ten']} + {pk['ten']} ({sk['ten']})",
                    }


if not CHAY_HANG_LOAT:
    print("CHAY_HANG_LOAT = False -> bỏ qua, chỉ đếm thử số tổ hợp sẽ chạy.")
    so_luong_uoc_tinh = sum(1 for _ in sinh_tat_ca_to_hop())
    print(f"Ước tính sẽ có khoảng {so_luong_uoc_tinh} tổ hợp hợp lệ (trước khi trừ tổ hợp đã tồn tại).")
else:
    tat_ca = list(sinh_tat_ca_to_hop())
    if GIOI_HAN_SO_LUONG:
        tat_ca = tat_ca[:GIOI_HAN_SO_LUONG]
    print(f"Chuẩn bị chạy {len(tat_ca)} tổ hợp...")

    ket_qua_hang_loat = []
    so_moi = so_da_co = so_loi = 0
    for i, th in enumerate(tat_ca, 1):
        kq = goi_sinh_to_hop(th)
        ket_qua_hang_loat.append(kq)
        if kq.get("loi"):
            so_loi += 1
            print(f"[{i}/{len(tat_ca)}] LỖI - {kq['nhan']}: {kq['loi']}")
        elif kq.get("da_ton_tai"):
            so_da_co += 1
        else:
            so_moi += 1
            print(f"[{i}/{len(tat_ca)}] Đã sinh ảnh mới - {kq['nhan']}")
        time.sleep(DO_TRE_GIUA_2_LUOT_GIAY)

    print(f"\nXong. Mới sinh: {so_moi} | Đã có sẵn (bỏ qua): {so_da_co} | Lỗi: {so_loi}")


CHAY_HANG_LOAT = False -> bỏ qua, chỉ đếm thử số tổ hợp sẽ chạy.


ReadTimeout: HTTPConnectionPool(host='localhost', port=3000): Read timed out. (read timeout=10)

KeyboardInterrupt: 